# 🧠 Word2Vec — Training on a Larger Corpus

## Objectives
- Build and preprocess a realistic 20-sentence product/tech review corpus
- Train Word2Vec using two architectures: **CBOW** and **Skip-gram**
- Explore vocabulary, most-similar words, and cosine similarity
- Compare CBOW vs Skip-gram embedding quality

**Library:** Gensim `Word2Vec`

## 📖 Background

**Word2Vec** learns dense vector representations of words by training a shallow neural network
to predict words from their context.

| Architecture | Mechanism | Strength |
|---|---|---|
| **CBOW** (Continuous Bag of Words) | Predicts target word from surrounding context words | Faster, works well for frequent words |
| **Skip-gram** | Predicts surrounding context words from a target word | Slower, but richer embeddings for rare words |

### Key hyperparameters
| Parameter | Meaning |
|---|---|
| `vector_size` | Dimensionality of each word vector |
| `window` | Number of context words on each side |
| `min_count` | Ignore words that appear fewer than N times |
| `epochs` | Number of full passes over the corpus during training |

---
## ✏️ Exercise 2: Train Word2Vec on a Larger Corpus

We build a 20-sentence product/tech review corpus, preprocess it fully,
then train and compare two Word2Vec architectures:

| Architecture | How it works | Best for |
|---|---|---|
| **CBOW** | Predicts target word from surrounding context | Frequent words, faster training |
| **Skip-gram** | Predicts context words from a target word | Rare words, richer embeddings |

In [25]:
import re, string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 20-sentence product/tech review corpus
raw_corpus = [
    "The smartphone has an incredible display with vivid colors and sharp resolution.",
    "Battery life is outstanding and lasts more than two days on a single charge.",
    "The camera produces stunning photographs even in low light conditions.",
    "Customer service was very helpful and resolved my issue within minutes.",
    "The laptop keyboard feels comfortable and typing experience is smooth.",
    "Delivery was fast and the packaging was secure preventing any damage.",
    "The sound quality of these headphones is rich, deep, and immersive.",
    "Screen resolution on this monitor is crystal clear and eye friendly.",
    "The gaming performance of this GPU is exceptional at high settings.",
    "This smartwatch tracks fitness data accurately including heart rate and sleep.",
    "Software updates are frequent and the operating system runs smoothly.",
    "The build quality feels premium with a solid aluminum chassis design.",
    "Connectivity options include USB-C, HDMI, and fast wireless Bluetooth.",
    "The streaming service offers thousands of movies and series in HD quality.",
    "Setup was straightforward and the user manual explains every step clearly.",
    "Price is reasonable compared to competitors offering similar specifications.",
    "The touchscreen response is highly accurate with minimal input latency.",
    "Voice assistant integration works seamlessly with smart home devices.",
    "Return policy is flexible and refunds are processed within three days.",
    "Overall this product exceeded my expectations and I highly recommend it.",
]

print(f"Corpus size: {len(raw_corpus)} sentences")

Corpus size: 20 sentences


In [26]:
# Full preprocessing pipeline: lowercase -> digits -> punctuation -> tokenize -> stopwords -> lemmatize
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in stop_words and len(w) > 1]
    return tokens

tokenized_corpus = [preprocess(sent) for sent in raw_corpus]

print("Tokenized corpus (first 5 sentences):")
for i, t in enumerate(tokenized_corpus[:5]):
    print(f"  {i+1}: {t}")

total_tokens = sum(len(s) for s in tokenized_corpus)
print(f"\nTotal sentences : {len(tokenized_corpus)}")
print(f"Total tokens    : {total_tokens}")

Tokenized corpus (first 5 sentences):
  1: ['smartphone', 'incredible', 'display', 'vivid', 'color', 'sharp', 'resolution']
  2: ['battery', 'life', 'outstanding', 'last', 'two', 'day', 'single', 'charge']
  3: ['camera', 'produce', 'stunning', 'photograph', 'even', 'low', 'light', 'condition']
  4: ['customer', 'service', 'helpful', 'resolved', 'issue', 'within', 'minute']
  5: ['laptop', 'keyboard', 'feel', 'comfortable', 'typing', 'experience', 'smooth']

Total sentences : 20
Total tokens    : 146


In [27]:
# Train CBOW model (sg=0)
# vector_size=100 : each word encoded as a 100-dim vector
# window=5        : context window of 5 words on each side
# epochs=100      : number of training passes over the corpus
cbow_model = Word2Vec(
    tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=100,
    seed=42,
    workers=1
)

# Train Skip-gram model (sg=1)
sg_model = Word2Vec(
    tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=100,
    seed=42,
    workers=1
)

vocab = sorted(cbow_model.wv.index_to_key)
print(f"Vocabulary size : {len(vocab)}")
print(f"Sample words    : {vocab[:20]}")

Vocabulary size : 137
Sample words    : ['accurate', 'accurately', 'aluminum', 'assistant', 'battery', 'bluetooth', 'build', 'camera', 'charge', 'chassis', 'clear', 'clearly', 'color', 'comfortable', 'compared', 'competitor', 'condition', 'connectivity', 'crystal', 'customer']


In [28]:
# Most similar words — CBOW vs Skip-gram
# Skip-gram generally captures richer semantic relationships
# because it trains on more prediction tasks per sentence.

probe_words = ['camera', 'battery', 'quality', 'screen']

print("Most Similar Words - CBOW (sg=0)")
print("-" * 55)
for word in probe_words:
    similar = cbow_model.wv.most_similar(word, topn=3)
    pairs = ', '.join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:<12} -> {pairs}")

print("\nMost Similar Words - Skip-gram (sg=1)")
print("-" * 55)
for word in probe_words:
    similar = sg_model.wv.most_similar(word, topn=3)
    pairs = ', '.join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:<12} -> {pairs}")

Most Similar Words - CBOW (sg=0)
-------------------------------------------------------
  camera       -> refund(0.32), update(0.30), input(0.30)
  battery      -> every(0.31), step(0.30), straightforward(0.29)
  quality      -> connectivity(0.34), processed(0.33), compared(0.32)
  screen       -> refund(0.37), crystal(0.29), chassis(0.29)

Most Similar Words - Skip-gram (sg=1)
-------------------------------------------------------
  camera       -> photograph(0.71), low(0.70), refund(0.68)
  battery      -> every(0.68), step(0.66), straightforward(0.65)
  quality      -> every(0.79), explains(0.78), premium(0.78)
  screen       -> refund(0.73), photograph(0.68), assistant(0.68)


In [29]:
# Cosine similarity between semantically related word pairs
# Values range from -1 (opposite) to +1 (identical direction)

pairs = [
    ('camera',  'photograph'),
    ('battery', 'charge'),
    ('screen',  'display'),
    ('quality', 'premium'),
]

print(f"{'Word 1':<15} {'Word 2':<15} {'CBOW':>8} {'Skip-gram':>10}")
print("-" * 52)
for w1, w2 in pairs:
    cbow_sim = cbow_model.wv.similarity(w1, w2)
    sg_sim   = sg_model.wv.similarity(w1, w2)
    print(f"  {w1:<13} {w2:<15} {cbow_sim:>8.4f} {sg_sim:>10.4f}")

Word 1          Word 2          CBOW  Skip-gram
----------------------------------------------------
  camera        photograph       0.2288     0.7098
  battery       charge           0.0092     0.5873
  screen        display         -0.0753     0.5214
  quality       premium          0.2771     0.7761


In [30]:
# Inspect the learned vector for 'quality'
# Each word is represented as a dense 100-dimensional float vector.

vec = cbow_model.wv['quality']
print(f"Word          : 'quality'")
print(f"Vector dims   : {len(vec)}")
print(f"First 10 dims : {np.round(vec[:10], 6).tolist()}")
print(f"Min value     : {vec.min():.6f}")
print(f"Max value     : {vec.max():.6f}")
print(f"L2 norm       : {np.linalg.norm(vec):.6f}")

Word          : 'quality'
Vector dims   : 100
First 10 dims : [-0.006059, 0.006094, -0.00076, -0.002067, -0.005022, 0.00831, -0.007642, 0.005077, -0.01028, -0.009991]
Min value     : -0.017424
Max value     : 0.019704
L2 norm       : 0.078696


### Key Observations

| Observation | Detail |
|---|---|
| **Skip-gram > CBOW** | Skip-gram yields higher similarity scores (~0.7) vs CBOW (~0.2) on this small corpus |
| **Corpus size matters** | With only 20 sentences, vectors are noisy — a 10k+ sentence corpus would yield much more meaningful neighbours |
| **vector_size=100** | Each word is a 100-dim vector; larger sizes capture more nuance but need more data |
| **epochs=100** | More passes help on small corpora where each sentence is seen fewer times |